# AgriMatch M2 -- NASA POWER Climate Backfill

> **Optimised for Colab L4 GPU runtime**

This notebook fetches daily climate data from the NASA POWER API for all
260 Ghana districts from **2006-01-01 to 2023-07-15** and saves the results
as `nasa_power_daily.csv` for import into your local PostgreSQL database.

## Before you start

- **NASA POWER API is free** -- no API key required.
- **Do not close this browser tab** while running.
- **Expected runtime**: 45-60 minutes with 16 parallel workers.
- **Restartable**: progress is saved to `/content/nasa_progress.csv` after
  every 20 completed district-chunks. Re-run all cells to resume -- already
  fetched chunks are skipped automatically.

## What this produces

| Column | Description |
|---|---|
| `obs_date` | Observation date (YYYY-MM-DD) |
| `district_id` | Sequential GADM district ID (1-260) |
| `district_name` | GADM NAME_2 district name |
| `region_name` | GADM NAME_1 region name |
| `temp_mean` | Mean 2m air temperature (C) |
| `temp_max` | Maximum 2m temperature (C) |
| `temp_min` | Minimum 2m temperature (C) |
| `solar_mj` | All-sky surface solar radiation (MJ/m2/day) |
| `humidity_pct` | Relative humidity at 2m (%) |
| `wind_ms` | Wind speed at 2m (m/s) |
| `et0_mm` | Reference ET0 by FAO Penman-Monteith (mm/day) |


In [ ]:
!pip install requests pandas numpy geopandas --quiet

import math
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, timedelta
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests

print('Imports OK')

In [ ]:
# Cell 3 -- District setup
# Downloads GADM Ghana level-2 boundaries if not already present,
# builds a GeoDataFrame with centroid_lat, centroid_lon, district_id 1-260.

GADM_URL  = 'https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_GHA.gpkg'
GADM_PATH = Path('/content/gadm41_GHA.gpkg')


def load_ghana_districts():
    if not GADM_PATH.exists():
        print('Downloading GADM Ghana boundaries (~25 MB) ...')
        resp = requests.get(GADM_URL, stream=True, timeout=(15, 300))
        resp.raise_for_status()
        downloaded = 0
        with GADM_PATH.open('wb') as fh:
            for chunk in resp.iter_content(chunk_size=65536):
                if chunk:
                    fh.write(chunk)
                    downloaded += len(chunk)
        print(f'  Downloaded {downloaded / 1_048_576:.1f} MB')
    else:
        print(f'GADM file already present: {GADM_PATH}')

    gdf = gpd.read_file(GADM_PATH, layer='ADM_ADM_2')
    gdf = gdf[['GID_2', 'NAME_1', 'NAME_2', 'geometry']].copy()
    gdf = gdf.rename(columns={'NAME_1': 'region_name', 'NAME_2': 'district_name'})
    gdf = gdf.to_crs('EPSG:4326')

    centroids           = gdf.geometry.centroid
    gdf['centroid_lat'] = centroids.y.round(6)
    gdf['centroid_lon'] = centroids.x.round(6)

    gdf = gdf.sort_values('GID_2').reset_index(drop=True)
    gdf['district_id'] = gdf.index + 1

    n_regions = gdf['region_name'].nunique()
    print(f'Loaded {len(gdf)} districts across {n_regions} regions')
    print(gdf[['district_id', 'district_name', 'region_name', 'centroid_lat', 'centroid_lon']].head(5).to_string(index=False))
    return gdf


gdf_districts = load_ghana_districts()

In [ ]:
# Cell 4 -- NASA POWER fetch function

_NASA_URL = 'https://power.larc.nasa.gov/api/temporal/daily/point'
_PARAMS   = 'T2M,T2M_MAX,T2M_MIN,ALLSKY_SFC_SW_DWN,RH2M,WS2M'
_MISSING  = -999.0

_RENAME = {
    'T2M':               'temp_mean',
    'T2M_MAX':           'temp_max',
    'T2M_MIN':           'temp_min',
    'ALLSKY_SFC_SW_DWN': 'solar_mj',
    'RH2M':              'humidity_pct',
    'WS2M':              'wind_ms',
}


def fetch_district_chunk(district_id, district_name, region_name, lat, lon, start_date, end_date):
    params = {
        'parameters': _PARAMS,
        'community':  'AG',
        'longitude':  lon,
        'latitude':   lat,
        'start':      start_date.strftime('%Y%m%d'),
        'end':        end_date.strftime('%Y%m%d'),
        'format':     'JSON',
        'user':       'agrimatch',
    }
    try:
        resp = requests.get(_NASA_URL, params=params, timeout=(10, 120))
        if not resp.ok:
            print(f'  WARN [{district_name}] HTTP {resp.status_code}: {resp.text[:200]}')
            return pd.DataFrame()

        payload    = resp.json()
        param_data = payload['properties']['parameter']

        df = pd.DataFrame(param_data)
        df.replace(_MISSING, float('nan'), inplace=True)
        df.index      = pd.to_datetime(df.index, format='%Y%m%d').date
        df.index.name = 'obs_date'
        df = df.reset_index()

        df['district_id']   = district_id
        df['district_name'] = district_name
        df['region_name']   = region_name
        df = df.rename(columns=_RENAME)

        cols = [
            'obs_date', 'district_id', 'district_name', 'region_name',
            'temp_mean', 'temp_max', 'temp_min', 'solar_mj', 'humidity_pct', 'wind_ms',
        ]
        return df[cols]

    except Exception as exc:
        print(f'  ERROR [{district_name}] {start_date} to {end_date}: {exc}')
        return pd.DataFrame()


print('fetch_district_chunk defined OK')

In [ ]:
# Cell 5 -- ET0 calculation (FAO Penman-Monteith, copied from nasa_power_client.py)

def calculate_et0(row):
    T    = row['temp_mean']
    Tmax = row['temp_max']
    Tmin = row['temp_min']
    Rs   = row['solar_mj']
    RH   = row['humidity_pct']
    u2   = row['wind_ms']

    if any(v != v for v in [T, Tmax, Tmin, Rs, RH, u2]):  # NaN check
        return float('nan')

    # Saturation vapour pressure (kPa)
    es = 0.6108 * (
        math.exp(17.27 * Tmax / (Tmax + 237.3))
        + math.exp(17.27 * Tmin / (Tmin + 237.3))
    ) / 2

    # Actual vapour pressure (kPa)
    ea = es * RH / 100

    # Slope of vapour pressure curve (kPa/C)
    delta = (
        4098 * (0.6108 * math.exp(17.27 * T / (T + 237.3)))
        / (T + 237.3) ** 2
    )

    # Psychrometric constant (kPa/C)
    gamma = 0.067

    # Net radiation approximation (MJ/m2/day)
    Rn = 0.77 * Rs - 2.1

    # FAO PM equation (mm/day)
    numerator   = 0.408 * delta * Rn + gamma * (900 / (T + 273)) * u2 * (es - ea)
    denominator = delta + gamma * (1 + 0.34 * u2)

    et0 = numerator / denominator
    return round(max(et0, 0.0), 3)


def apply_et0(df):
    if df.empty:
        return df
    df = df.copy()
    df['et0_mm'] = df.apply(calculate_et0, axis=1)
    return df


# Smoke test
import pandas as _pd
_test = _pd.Series({'temp_mean': 28.0, 'temp_max': 34.0, 'temp_min': 22.0,
                    'solar_mj': 18.5, 'humidity_pct': 65.0, 'wind_ms': 1.8})
print(f'calculate_et0 and apply_et0 defined OK  |  smoke test ET0 = {calculate_et0(_test)} mm/day (expect 4-6)')

In [ ]:
# Cell 6 -- Date chunking (NASA POWER hard limit: 366 days per request)

_MAX_DAYS = 366


def chunk_date_range(start_date, end_date, max_days=_MAX_DAYS, verbose=True):
    chunks = []
    cur = start_date
    while cur <= end_date:
        chunk_end = min(cur + timedelta(days=max_days - 1), end_date)
        chunks.append((cur, chunk_end))
        cur = chunk_end + timedelta(days=1)
    if verbose:
        print(f'Date range {start_date} to {end_date} -> {len(chunks)} chunk(s):')
        for cs, ce in chunks:
            print(f'  {cs} to {ce}  ({(ce - cs).days + 1} days)')
    return chunks


print('chunk_date_range defined OK')

In [ ]:
# Cell 7 -- Progress save and restore

_PROGRESS_CSV = Path('/content/nasa_progress.csv')


def save_progress(df, path=_PROGRESS_CSV):
    if df.empty:
        return
    df.to_csv(path, index=False)
    print(f'  [progress] Saved {len(df):,} rows to {path.name}')


def load_progress(path=_PROGRESS_CSV):
    if not path.exists():
        print('No existing progress file -- starting fresh.')
        return pd.DataFrame()
    df = pd.read_csv(path)
    df['obs_date'] = pd.to_datetime(df['obs_date']).dt.date
    df = df.drop_duplicates(subset=['district_id', 'obs_date'])
    n_dists = df['district_id'].nunique()
    n_dates = df['obs_date'].nunique()
    print(f'Loaded progress: {len(df):,} rows | {n_dists} districts | {n_dates} unique dates')
    return df


print('save_progress and load_progress defined OK')

In [ ]:
# Cell 8 -- Main backfill function

def run_nasa_backfill(districts_gdf, start_date, end_date, max_workers=16, delay=0.3):
    progress_df = load_progress()
    chunks      = chunk_date_range(start_date, end_date, verbose=False)

    # Build all (district, chunk) task tuples
    all_tasks = []
    for _, row in districts_gdf.iterrows():
        for cs, ce in chunks:
            all_tasks.append((
                int(row.district_id), row.district_name, row.region_name,
                float(row.centroid_lat), float(row.centroid_lon),
                cs, ce,
            ))

    # Filter out already-complete chunks
    pending = []
    if progress_df.empty:
        pending = list(all_tasks)
    else:
        for task in all_tasks:
            did, dname, rname, lat, lon, cs, ce = task
            expected = (ce - cs).days + 1
            mask = (
                (progress_df['district_id'] == did)
                & (progress_df['obs_date'] >= cs)
                & (progress_df['obs_date'] <= ce)
            )
            done = int(progress_df.loc[mask, 'obs_date'].nunique())
            if done < expected:
                pending.append(task)

    skipped = len(all_tasks) - len(pending)
    print(f'Total district-chunks : {len(all_tasks):,}')
    print(f'Already complete      : {skipped:,}')
    print(f'To fetch              : {len(pending):,}')

    if not pending:
        print('Nothing to fetch.')
        return progress_df

    def fetch_task(task):
        did, dname, rname, lat, lon, cs, ce = task
        time.sleep(delay)
        df = fetch_district_chunk(did, dname, rname, lat, lon, cs, ce)
        if not df.empty:
            df = apply_et0(df)
        return df

    new_frames = []
    completed  = 0
    t_start    = time.time()
    save_every = 20

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(fetch_task, task): task for task in pending}
        for future in as_completed(futures):
            result_df = future.result()
            completed += 1
            if result_df is not None and not result_df.empty:
                new_frames.append(result_df)

            if completed % save_every == 0 and new_frames:
                frames = ([progress_df] if not progress_df.empty else []) + new_frames
                save_progress(pd.concat(frames, ignore_index=True))

            if completed % 20 == 0 or completed == len(pending):
                elapsed = time.time() - t_start
                rate    = completed / elapsed if elapsed > 0 else 0
                rem     = (len(pending) - completed) / rate if rate > 0 else 0
                print(
                    f'  Completed {completed} / {len(pending)} district-chunks | '
                    f'Elapsed: {elapsed/60:.1f}m | Est: {rem/60:.1f}m'
                )

    all_frames = ([progress_df] if not progress_df.empty else []) + new_frames
    result = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
    save_progress(result)
    return result


print('run_nasa_backfill defined OK')


In [ ]:
# Cell 9 -- Run the full backfill

FULL_START = date(2006, 1, 1)
FULL_END   = date(2023, 7, 15)
OUTPUT_CSV = Path('/content/nasa_power_daily.csv')

# Preview the chunk plan before the long run
chunks   = chunk_date_range(FULL_START, FULL_END)
n_tasks  = len(chunks) * len(gdf_districts)
n_days   = (FULL_END - FULL_START).days + 1
exp_rows = len(gdf_districts) * n_days
print(f'\nEstimated API calls  : {n_tasks:,}  ({len(gdf_districts)} districts x {len(chunks)} chunks)')
print(f'Expected output rows : {exp_rows:,}  ({len(gdf_districts)} districts x {n_days} days)')
print()

result_df = run_nasa_backfill(
    gdf_districts,
    FULL_START,
    FULL_END,
    max_workers=16,
    delay=0.3,
)

# Safety check: apply ET0 if somehow missing (should already be applied per chunk)
if not result_df.empty and 'et0_mm' not in result_df.columns:
    print('Applying ET0 to full result ...')
    result_df = apply_et0(result_df)

result_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved {len(result_df):,} rows to {OUTPUT_CSV}')
print(f'Expected approximately 1,665,210 rows (260 districts x 6,393 days)')


In [ ]:
# Cell 10 -- Validation and download

if not OUTPUT_CSV.exists():
    print('ERROR: nasa_power_daily.csv not found. Run Cell 9 first.')
else:
    df = pd.read_csv(OUTPUT_CSV)

    date_min    = df['obs_date'].min()
    date_max    = df['obs_date'].max()
    n_dates     = df['obs_date'].nunique()
    n_districts = df['district_name'].nunique()

    print()
    print('=' * 65)
    print('FINAL SUMMARY -- nasa_power_daily.csv')
    print('=' * 65)
    print(f'  Total rows       : {len(df):,}')
    print(f'  Date range       : {date_min} to {date_max}')
    print(f'  Unique dates     : {n_dates:,}')
    print(f'  Unique districts : {n_districts}')
    print()

    # ET0 by region -- northern Ghana should be highest (more arid, higher radiation)
    if 'et0_mm' in df.columns:
        print('  Average ET0 by region (mm/day) -- northern Ghana should be highest:')
        et0_by_region = (
            df.groupby('region_name')['et0_mm']
            .mean()
            .round(3)
            .sort_values(ascending=False)
        )
        north_regions = {'Upper East', 'Upper West', 'Northern', 'Savannah', 'North East'}
        south_regions = {'Greater Accra', 'Central', 'Western', 'Volta'}
        for region, val in et0_by_region.items():
            tag = ' <-- north' if region in north_regions else ''
            print(f'    {region:<30} {val:.3f}{tag}')
        n_idx = [r for r in north_regions if r in et0_by_region.index]
        s_idx = [r for r in south_regions if r in et0_by_region.index]
        n_et0 = et0_by_region[n_idx].mean() if n_idx else float('nan')
        s_et0 = et0_by_region[s_idx].mean() if s_idx else float('nan')
        chk   = 'PASS' if n_et0 > s_et0 else 'WARN'
        print(f'\n  [{chk}] North avg ET0 {n_et0:.3f} vs South avg ET0 {s_et0:.3f}')

    # Districts with missing dates
    print()
    n_expected       = (FULL_END - FULL_START).days + 1
    dist_date_counts = df.groupby('district_name')['obs_date'].nunique()
    incomplete       = dist_date_counts[dist_date_counts < n_expected]
    if len(incomplete) > 0:
        print(f'  Districts with missing dates ({len(incomplete)}):')
        for dname, cnt in incomplete.sort_values().items():
            print(f'    {dname}: {cnt} / {n_expected} dates')
    else:
        print(f'  All {n_districts} districts have full date coverage.')

    print()
    print('Downloading nasa_power_daily.csv ...')
    from google.colab import files
    files.download('/content/nasa_power_daily.csv')

## Importing into local PostgreSQL after download

After downloading `nasa_power_daily.csv`, run the following script from your
project root to import the data into `nasa_power_daily`.

The script matches rows by `district_name` against `ghana_districts.district_name`
to resolve local `district_id` values (which may differ from the sequential
GADM IDs used in Colab).

### Step 1 -- Copy the CSV to your project

Move `nasa_power_daily.csv` into your project root (or any convenient path).

### Step 2 -- Run the import script

Save as `setup/import_nasa_power_csv.py` and run from the project root:

```python
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent.parent))

import pandas as pd
from db.connection import get_session
from sqlalchemy import text

CSV_PATH = Path('nasa_power_daily.csv')  # adjust path if needed

df = pd.read_csv(CSV_PATH, parse_dates=['obs_date'])
df['obs_date'] = df['obs_date'].dt.date
print(f'Loaded {len(df):,} rows from {CSV_PATH}')

# Resolve district_name -> local district_id
with get_session() as s:
    id_rows = s.execute(text('SELECT id, district_name FROM ghana_districts')).all()
name_to_id = {r.district_name: r.id for r in id_rows}

df['district_id'] = df['district_name'].map(name_to_id)
unmatched = df[df['district_id'].isna()]['district_name'].unique()
if len(unmatched) > 0:
    print(f'WARNING: {len(unmatched)} district names did not match local DB:')
    for n in sorted(unmatched):
        print(f'  {n}')

df = df.dropna(subset=['district_id'])
df['district_id'] = df['district_id'].astype(int)

db_cols = ['obs_date', 'district_id', 'temp_mean', 'temp_max', 'temp_min',
           'solar_mj', 'humidity_pct', 'wind_ms', 'et0_mm']
records = df[db_cols].to_dict(orient='records')

BATCH = 5000
total_inserted = 0
for i in range(0, len(records), BATCH):
    batch = records[i : i + BATCH]
    with get_session() as s:
        before = s.execute(text('SELECT COUNT(*) FROM nasa_power_daily')).scalar()
        s.execute(text('''
            INSERT INTO nasa_power_daily
                (obs_date, district_id, temp_mean, temp_max, temp_min,
                 solar_mj, humidity_pct, wind_ms, et0_mm)
            VALUES
                (:obs_date, :district_id, :temp_mean, :temp_max, :temp_min,
                 :solar_mj, :humidity_pct, :wind_ms, :et0_mm)
            ON CONFLICT (obs_date, district_id) DO NOTHING
        '''), batch)
        after = s.execute(text('SELECT COUNT(*) FROM nasa_power_daily')).scalar()
    total_inserted += after - before
    print(f'  Batch {i//BATCH + 1}: inserted {after - before} rows (total {total_inserted:,})')

print(f'Import complete: {total_inserted:,} rows inserted into nasa_power_daily')
```

### Step 3 -- Verify

```sql
SELECT
    COUNT(*)                          AS total_rows,
    MIN(obs_date)                     AS date_min,
    MAX(obs_date)                     AS date_max,
    COUNT(DISTINCT district_id)       AS districts,
    ROUND(AVG(et0_mm)::numeric, 3)    AS avg_et0_mm
FROM nasa_power_daily;
```

Expected: ~1.66M rows, 2006-01-01 to 2023-07-15, 260 districts, avg ET0 around 4-5 mm/day.

### Note on district name matching

GADM NAME_2 names may differ slightly from the district names in your local
`ghana_districts` table (which was loaded from a different boundary source).
If the import script reports unmatched districts, check the names manually and
add entries to `name_to_id` before the insert loop:

```python
# Example manual overrides:
name_to_id['Kumasi'] = name_to_id.get('Kumasi Metropolitan', name_to_id.get('Kumasi'))
```
